In [ ]:
# ==========================================
# CELL 1: ALL IMPORTS
# ==========================================
# Data Manipulation & Visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import ScalarFormatter

# Preprocessing & Core ML Tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, silhouette_score

# Supervised Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB

# Unsupervised Models
from sklearn.cluster import KMeans, DBSCAN

# Environment settings
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

In [ ]:
# ==========================================
# CELL 2: DATA SPLITTING & SCALING
# ==========================================
file_path = "Input_Data/01_cleaned_data/uwb_dataset_classification.parquet"

df = pd.read_parquet(file_path) 
X = df.drop('NLOS', axis=1)
y = df['NLOS']

# 1. Split: 80% Train, 20% Test (Balanced)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 2. Scale: Fit ONLY on training data, transform both
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Data Pipeline Ready!")
print(f"Training Features: {X_train.shape} | Testing Features: {X_test.shape}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

def plot_independent_dashboard(model_name, grid, param_name):
    # Extract results
    best_model = grid.best_estimator_
    best_param_val = grid.best_params_[param_name]
    cv_res = pd.DataFrame(grid.cv_results_)
    
    # Get predictions only for the best model for Matrix and ROC
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]
    
    # --- UNIVERSAL FIX 1: Handle "None" / NaN values safely ---
    raw_params = cv_res[f'param_{param_name}'].values
    valid_nums = [x for x in raw_params if pd.notna(x)] # Filter out None/NaN
    
    # Create a safe placeholder for 'None' so math doesn't crash
    if valid_nums:
        placeholder = max(valid_nums) + 5 if max(valid_nums) >= 1 else max(valid_nums) * 1.5
    else:
        placeholder = 1
        
    numeric_params = [placeholder if pd.isna(x) else float(x) for x in raw_params]
    labels = ["None" if pd.isna(x) else str(x) for x in raw_params]
    
    # Create Layout
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    best_label = "None" if pd.isna(best_param_val) else best_param_val
    fig.suptitle(f"Model Analysis: {model_name} | Best {param_name}: {best_label}", 
                 fontsize=18, fontweight='bold', y=1.05)
    
    # Graph 1: Accuracy Flow (All Hyperparameters)
    axes[0].plot(numeric_params, cv_res['mean_train_score'], marker='s', label='Train Accuracy', color='skyblue', lw=2)
    axes[0].plot(numeric_params, cv_res['mean_test_score'], marker='o', label='Test Accuracy (CV)', color='salmon', lw=2)
    
    # --- UNIVERSAL FIX 2: Auto-Detect Scale ---
    # Log scale is only used for parameters that span magnitudes (C, alpha, gamma, learning_rate)
    # Linear scale is used for integers (n_neighbors, max_depth, n_estimators)
    log_params = ['C', 'alpha', 'gamma', 'learning_rate']
    
    if param_name in log_params:
        axes[0].set_xscale('log') 
        axes[0].xaxis.set_major_formatter(ScalarFormatter())
    else:
        axes[0].set_xscale('linear')
        
    # Apply the safe ticks and labels
    axes[0].set_xticks(numeric_params)
    axes[0].set_xticklabels(labels, rotation=45)
    
    axes[0].set_title(f"Accuracy Curve (All {param_name})")
    axes[0].set_xlabel(f"Hyperparameter: {param_name}")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    
    # Graph 2: Confusion Matrix (Best Hyperparameter Only)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm).plot(ax=axes[1], cmap='Blues', colorbar=False)
    axes[1].set_title(f"Confusion Matrix (Best {param_name}={best_label})")
    
    # Graph 3: ROC Curve (Best Hyperparameter Only)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[2].plot(fpr, tpr, color='darkorange', label=f'AUC = {auc(fpr, tpr):.3f}')
    axes[2].plot([0, 1], [0, 1], color='navy', linestyle='--')
    axes[2].set_title(f"ROC Curve (Best {param_name}={best_label})")
    axes[2].legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

param_grid = {'C': [0.0005,0.0007,0.0009, 0.001, 0.002, 0.004, 0.006, 0.008, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1]}

# 1. Define grid_lr
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42), 
    param_grid, 
    cv=3, 
    return_train_score=True, 
    n_jobs=-1
)

# 2. Fit using grid_lr
grid_lr.fit(X_train, y_train)

# 3. Plot using grid_lr
plot_independent_dashboard("Logistic Regression", grid_lr, 'C')

In [ ]:
param_grid = {'n_neighbors': [25,26, 27, 28,29,30,31,32,33,34,35]}

# Define grid_knn (using a unique variable name to prevent overwriting)
grid_knn = GridSearchCV(
    KNeighborsClassifier(), 
    param_grid, 
    cv=3, 
    return_train_score=True, 
    n_jobs=-1
)

# Fit the model
grid_knn.fit(X_train, y_train)

# Output results using our specific KNN plotting function
plot_independent_dashboard("K-Nearest Neighbors", grid_knn, 'n_neighbors')

In [ ]:
from sklearn.tree import DecisionTreeClassifier
param_grid = {'max_depth': [5,10, 15,20, 30, None]}
grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=3, return_train_score=True, n_jobs=-1)
grid_dt.fit(X_train, y_train)
plot_independent_dashboard("Decision Tree", grid_dt, 'max_depth')



In [ ]:
from sklearn.ensemble import RandomForestClassifier
param_grid = {'n_estimators': [5,10,15,20,30,40, 50, 100]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid, cv=3, return_train_score=True, n_jobs=-1)
grid_rf.fit(X_train, y_train)
plot_independent_dashboard("Random Forest", grid_rf, 'n_estimators')

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
param_grid = {'n_estimators': [10, 50, 100]}
grid_gb = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid, cv=3, return_train_score=True, n_jobs=-1)
grid_gb.fit(X_train, y_train)
plot_independent_dashboard("Gradient Boosting", grid_gb, 'n_estimators')

In [ ]:
from sklearn.neural_network import MLPClassifier
param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]}
grid_nn = GridSearchCV(MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42), 
                    param_grid, cv=3, return_train_score=True, n_jobs=-1)
grid_nn.fit(X_train, y_train)
plot_independent_dashboard("Neural Network", grid_nn, 'alpha')

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV

# Logarithmic range from very tiny (default is 1e-9) up to 1.0
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]}

grid_nb = GridSearchCV(
    GaussianNB(), 
    param_grid, 
    cv=3, 
    return_train_score=True, 
    n_jobs=-1
)

print("Training Naive Bayes...")
grid_nb.fit(X_train, y_train)

# Output results using your universal function
plot_independent_dashboard("Gaussian Naive Bayes", grid_nb, 'var_smoothing')

In [ ]:
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# 1. Apply PCA to keep 80% of the variance
pca_svm = PCA(n_components=0.80, random_state=42) 
X_train_SVM_PCA = pca_svm.fit_transform(X_train)
X_test_SVM_PCA = pca_svm.transform(X_test)

print("--- SVM-Specific PCA Complete ---")
print(f"Original features: {X_train.shape[1]}")
print(f"Reduced features for SVM: {X_train_SVM_PCA.shape[1]}")
print("Training SVM on reduced features... Please wait.")



In [ ]:
# 2. Setup the Grid Search for SVM
param_grid = {'C': [0.01, 0.1, 1, 10, 50]}

grid_svm_pca = GridSearchCV(
    SVC(probability=True, random_state=42), 
    param_grid, 
    cv=3, 
    return_train_score=True, 
    n_jobs=-1
)

# 3. Fit on the PCA-reduced training data
grid_svm_pca.fit(X_train_SVM_PCA, y_train)

# 4. THE JUPYTER VARIABLE HACK: 
# Temporarily assign the reduced test set to 'X_test' so the dashboard function
# uses the correct shape, then safely swap it back to the original!
X_test_original = X_test 
X_test = X_test_SVM_PCA 

plot_independent_dashboard("Support Vector Machine (80% PCA)", grid_svm_pca, 'C')

# Restore the original 131-feature X_test for your other models
X_test = X_test_original

In [ ]:
%pip install xgboost
# ==========================================
# CELL 14: XGBOOST (EXTREME GRADIENT BOOSTING)
# ==========================================
# Note: If you get a ModuleNotFoundError, run: !pip install xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

print("Training XGBoost...")

# Tuning the number of trees to find the peak performance without overfitting
param_grid = {'n_estimators': [10, 50, 100, 150, 200, 300]}

# Define the Grid Search
# tree_method='hist' is the secret weapon that makes XGBoost run incredibly fast
grid_xgb = GridSearchCV(
    XGBClassifier(random_state=42, tree_method='hist', n_jobs=-1), 
    param_grid, 
    cv=3, 
    return_train_score=True, 
    n_jobs=-1
)

# Fit the model to the scaled training data
grid_xgb.fit(X_train, y_train)

# Output results using the universal function
plot_independent_dashboard("XGBoost", grid_xgb, 'n_estimators')

In [ ]:
# Create a dictionary of your trained models
trained_models = {
    "K-Nearest Neighbors": grid_knn,
    "Decision Tree": grid,
    "Logistic Regression": grid_lr,
    "Random Forest": grid_rf,
    "Gradient Boosting": grid_gb,
    "Neural Network": grid_nn,
    "Naive Bayes": grid_nb,
    "SVM (with PCA)": grid_svm_pca,
    "XGBoost": grid_xgb
}

print("🏆 MODEL LEADERBOARD (By Cross-Validation Accuracy) 🏆")
print("-" * 55)

# Extract the best score from each and sort them
leaderboard = []
for name, model in trained_models.items():
    # .best_score_ is the highest Test Accuracy found during GridSearch
    leaderboard.append((name, model.best_score_ * 100))

# Sort from highest to lowest
leaderboard.sort(key=lambda x: x[1], reverse=True)

# Print the results
for rank, (name, score) in enumerate(leaderboard, 1):
    print(f"{rank}. {name:<25} | Score: {score:.2f}%")

In [ ]:
from sklearn.svm import SVC
# Testing a narrow C range to save time
param_grid = {'C': [0.01, 0.1, 1, 10, 50, 100]}
grid_svm = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=3, return_train_score=True, n_jobs=-1)
grid_svm.fit(X_train, y_train)
plot_independent_dashboard("SVM", grid_svm, 'C')